In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))
from annnet.core.graph import AnnNet

In [ ]:
# ---------- Setup ----------
G = AnnNet(directed=True)
conditions = ['Healthy', 'Stressed', 'Disease']
for c in conditions:
    G.slices.add_slice(c, condition=c)

# Entities
proteins = [f'P{i}' for i in range(1, 151)]  # P1..P150
transcripts = [f'T{i}' for i in range(1, 61)]  # T1..T60 (treat as vertices)
enz_edge_entities = [f'edge_rxn_{i}' for i in range(1, 11)]  # edge-entities for reactions

# Seed some vertex attributes
for p in proteins[:10]:
    G.add_vertices(p, slice='Healthy', family='kinase')
for p in proteins[10:]:
    G.add_vertices(p, slice='Healthy')
for t in transcripts:
    G.add_vertices(t, slice='Healthy', kind='transcript')
for ee in enz_edge_entities:
    G.add_edges(edge_id=ee, as_entity=True, slice='Healthy', role='enzyme')

# Propagate initial vertices to all slices
for lid in ['Stressed', 'Disease']:
    for vid in G.slices.vertices('Healthy'):
        G.slices.add_vertex_to_slice(lid, vid)

In [ ]:
# ---------- Build PPI edges in all slices ----------
import random


def rand_weight(base=1.0, jitter=0.5):
    return max(0.05, base + (random.random() - 0.5) * 2 * jitter)


ppis = []
for _ in range(320):
    u, v = random.sample(proteins, 2)
    w = rand_weight(1.2, 0.6)
    e = G.add_edges(u, v, slice='Healthy', weight=w, directed=False)
    ppis.append(e)

# Stress/disease slice variants (override per-slice weights)
for eid in ppis:
    # Stressed: mostly +10% with jitter
    G.slices.add_edge_to_slice('Stressed', eid)
    G.attrs.set_edge_slice_attrs(
        'Stressed', eid, weight=G.edge_weights[eid] * rand_weight(1.10, 0.1)
    )
    # Disease: some edges get weaker; others stronger
    G.slices.add_edge_to_slice('Disease', eid)
    factor = 0.7 if random.random() < 0.4 else 1.3
    G.attrs.set_edge_slice_attrs(
        'Disease', eid, weight=G.edge_weights[eid] * rand_weight(factor, 0.15)
    )

In [ ]:
# ---------- Complexes as undirected hyperedges ----------
complexes = []
for _ in range(12):
    members = set(random.sample(proteins, random.choice([3, 4, 5])))
    hid = G.add_edges(src=members, directed=False, slice='Healthy', weight=rand_weight(1.0, 0.2))
    complexes.append(hid)
    # complex exists in all slices (same membership)
    for lid in ['Stressed', 'Disease']:
        G.slices.add_edge_to_slice(lid, hid)

In [ ]:
# ---------- Directed signaling cascades as hyperedges ----------
cascades = []
while len(cascades) < 8:
    head = set(random.sample(proteins, random.choice([1, 2])))
    tail = set(random.sample(proteins, random.choice([2, 3, 4])))
    if head & tail:
        continue  # resample until disjoint
    hid = G.add_edges(
        src=head,
        tgt=tail,
        directed=True,
        name=f'test{random.choice([1, 20])}',
        slice='Healthy',
        weight=rand_weight(1.0, 0.4),
    )
    cascades.append(hid)
    for lid in ['Stressed', 'Disease']:
        G.slices.add_edge_to_slice(lid, hid)

In [ ]:
# ---------- Reactions connecting vertices to edge-entities ----------
for ee in enz_edge_entities:
    s, t = random.sample(proteins, 2)
    G.add_edges(s, ee, slice='Healthy', weight=1.0 + random.random())
    G.add_edges(ee, t, slice='Healthy', weight=1.0 + random.random())
    # propagate across slices
    for lid in ['Stressed', 'Disease']:
        real_edges = {
            e
            for e in G.slices.edges('Healthy')
            if e in G.edge_definitions or e in G.hyperedge_definitions
        }
        G.slices.add_edges(lid, real_edges)

In [ ]:
# ---------- MULTILAYER SETUP FOR BIOLOGICAL PPI ----------

# Define 3 aspects via the layers accessor
G.layers.set_aspects(['time', 'cell_type', 'condition'])
G.layers.set_elementary_layers(
    {
        'time': ['t0', 't1', 't2'],
        'cell_type': ['liver', 'neuron'],
        'condition': ['Healthy', 'Stressed', 'Disease'],
    }
)

# Assign attributes to aspects (metadata)
G.layers.set_aspect_attrs('time', unit='hours', description='Sampling time')
G.layers.set_aspect_attrs('cell_type', ontology='UBERON', modality='bulk')
G.layers.set_aspect_attrs('condition', perturbation='in-vitro assay')


# ----- Add ELEMENTARY-LAYER metadata -----
for t, hr in zip(['t0', 't1', 't2'], [0, 6, 24], strict=False):
    G.layers.set_elementary_layer_attrs('time', t, time_hours=hr)

G.layers.set_elementary_layer_attrs('cell_type', 'liver', tissue='hepatic')
G.layers.set_elementary_layer_attrs('cell_type', 'neuron', tissue='neural')

G.layers.set_elementary_layer_attrs('condition', 'Healthy', severity=0)
G.layers.set_elementary_layer_attrs('condition', 'Stressed', severity=1)
G.layers.set_elementary_layer_attrs('condition', 'Disease', severity=3)


# ----- Add full-layer attributes (aspect tuples) -----
import itertools

for L in itertools.product(
    ['t0', 't1', 't2'], ['liver', 'neuron'], ['Healthy', 'Stressed', 'Disease']
):
    t, cell, cond = L
    attrs = {
        'is_baseline': (t == 't0'),
        'is_neural_state': (cell == 'neuron'),
        'pathology_index': {'Healthy': 0, 'Stressed': 1, 'Disease': 2}[cond],
    }
    G.layers.set_layer_attrs(L, **attrs)


# ----- Set vertex–layer presence -----
for p in proteins:
    for L in G.layers.iter_layers():
        G.add_vertices(p, layer=L)

for t in transcripts:
    for L in G.layers.iter_layers():
        if L[0] in ['t0', 't2']:  # transcripts only measured baseline+late
            G.add_vertices(t, layer=L)

In [ ]:
# ----- Add INTER-layer edges (cross-time alignment) -----
for p in proteins[:50]:
    for cell in ['liver', 'neuron']:
        for cond in ['Healthy', 'Stressed', 'Disease']:
            L0 = ('t0', cell, cond)
            L1 = ('t1', cell, cond)
            L2 = ('t2', cell, cond)
            G.add_edges((p, L0), (p, L1), weight=0.5)
            G.add_edges((p, L1), (p, L2), weight=0.5)


# ----- Add COUPLING edges (cross-cell-type alignment) -----
for p in proteins[:30]:
    for t in ['t0', 't1', 't2']:
        for cond in ['Healthy', 'Stressed', 'Disease']:
            La = (t, 'liver', cond)
            Lb = (t, 'neuron', cond)
            G.add_edges((p, La), (p, Lb), weight=0.3)

print('✓ Multilayer PPI model injected.')

In [9]:
# ---------- Basic sanity ----------
print('vertices:', G.number_of_vertices(), 'Edges:', G.number_of_edges())

# Only true "vertices" are counted by number_of_vertices() (proteins + transcripts)
expected_vertices = len(set(proteins)) + len(set(transcripts))  # 150 + 60 = 210
assert G.number_of_vertices() >= expected_vertices, (
    f'Expected ≥{expected_vertices}, got {G.number_of_vertices()}'
)

# Edge-entities are tracked as entity_type == 'edge' (not included in number_of_vertices)
edge_entity_ids = set(enz_edge_entities)
edge_entity_count = sum(
    1 for _id, et in G.entity_types.items() if et == 'edge' and _id in edge_entity_ids
)
assert edge_entity_count == len(edge_entity_ids), (
    f'Expected {len(edge_entity_ids)} edge-entities, got {edge_entity_count}'
)

# Edges: PPIs (320) + complexes (12) + cascades (8) + reaction links (10*2) = 360 minimum
assert G.number_of_edges() >= 320 + 12 + 8 + (10 * 2)

vertices: 210 Edges: 1230


In [ ]:
# ---------- Views & top edges by condition ----------
import polars as pl

for cond in conditions:
    EV = G.views.edges(slice=cond, resolved_weight=True)
    print(f'[{cond}] edges_view rows =', EV.height)
    top = (
        EV.filter(pl.col('kind') == 'binary')
        .sort('effective_weight', descending=True)
        .select(['edge_id', 'source', 'target', 'effective_weight'])
        .head(5)
    )
    print(f'\nTop 5 binary edges by effective_weight in {cond}:')
    print(top)

In [ ]:
# ---------- slice analytics ----------
stats = G.slices.stats()
print('\nslice stats:', stats)

conserved = G.slices.conserved_edges(min_slices=3)  # present in all 3 conditions
print('\nConserved edges (in all conditions):', len(conserved))

disease_specific = G.slices.specific_edges('Disease')
print('Disease-specific edges:', len(disease_specific))

changes = G.slices.temporal_dynamics(['Healthy', 'Stressed', 'Disease'], metric='edge_change')
print('\nTemporal edge changes (Healthy→Stressed→Disease):', changes)
assert len(changes) == 2

In [ ]:
# ---------- Presence queries ----------
some_e = next(iter(G.edge_to_idx.keys()))
print('\nEdge presence for', some_e, ':', G.slices.edge_presence(edge_id=some_e))
some_p = random.choice(proteins)
print('vertex presence for', some_p, ':', G.slices.vertex_presence(some_p))

In [13]:
# ---------- Traversal checks ----------
q = random.choice(proteins)
print(f'\nNeighbors({q}) =>', G.neighbors(q)[:10])
print(f'Out({q}) =>', G.out_neighbors(q)[:10])
print(f'In({q}) =>', G.in_neighbors(q)[:10])


Neighbors(P97) => ['P67', 'P72', 'P70']
Out(P97) => ['P67', 'P72', 'P70']
In(P97) => ['P67', 'P72', 'P70']


In [14]:
# ---------- Subgraph slice ----------
H = G.subgraph_from_slice('Disease', resolve_slice_weights=True)
assert set(H.vertices()).issubset(set(G.vertices()))
assert set(H.edges()).issubset(set(G.edges()))
print('\nDisease subgraph: vertices =', H.number_of_vertices(), 'edges =', H.number_of_edges())


Disease subgraph: vertices = 210 edges = 360


In [ ]:
# ---------- copy ----------

K = G.ops.copy()
assert set(K.vertices()) == set(G.vertices())
assert set(K.edges()) == set(G.edges())
# hyperedge shape preserved
any_hyper = next(e for e, k in G.edge_kind.items() if k == 'hyper')
assert K.edge_kind.get(any_hyper) == 'hyper'
# slice sets preserved
for lid in G.slices.list_slices(include_default=True):
    assert K.slices.vertices(lid) == G.slices.vertices(lid)
    assert K.slices.edges(lid) == G.slices.edges(lid)
print('copy() OK')

In [16]:
# ---------- Remove operations stress ----------
to_drop_vertices = random.sample(proteins, 5)
for n in to_drop_vertices:
    if n in G.entity_to_idx:
        G.remove_vertices(n)
print(
    '\nAfter removing 5 proteins: vertices =',
    G.number_of_vertices(),
    'edges =',
    G.number_of_edges(),
)

to_drop_edges = list(G.edge_to_idx.keys())[:10]
for eid in to_drop_edges:
    if eid in G.edge_to_idx:
        G.remove_edges(eid)
print('After removing 10 edges: vertices =', G.number_of_vertices(), 'edges =', G.number_of_edges())


After removing 5 proteins: vertices = 205 edges = 1198
After removing 10 edges: vertices = 205 edges = 1188


In [ ]:
# ---------- Audit & memory ----------
audit = G.attrs.audit_attributes()
print('\nAudit:', audit)
mem_bytes = G.ops.memory_usage()
print('Approx memory usage (bytes):', int(mem_bytes))
assert mem_bytes > 0

print('\nReality-check finished ✅')

In [18]:
events = G.history()  # list[dict]
df = G.history(as_df=True)  # Polars DF [DataFrame]

In [19]:
print(df.head())
events[:3]

shape: (5, 10)
┌─────────┬──────────────┬──────────┬──────────────┬───┬────────┬───────────┬─────────┬────────────┐
│ version ┆ ts_utc       ┆ mono_ns  ┆ op           ┆ … ┆ result ┆ vertex_id ┆ slice   ┆ attributes │
│ ---     ┆ ---          ┆ ---      ┆ ---          ┆   ┆ ---    ┆ ---       ┆ ---     ┆ ---        │
│ i64     ┆ str          ┆ i64      ┆ str          ┆   ┆ str    ┆ str       ┆ str     ┆ struct[1]  │
╞═════════╪══════════════╪══════════╪══════════════╪═══╪════════╪═══════════╪═════════╪════════════╡
│ 1       ┆ 2025-12-03T1 ┆ 7807091  ┆ set_slice_at ┆ … ┆ null   ┆ null      ┆ null    ┆ null       │
│         ┆ 2:19:12.2516 ┆          ┆ trs          ┆   ┆        ┆           ┆         ┆            │
│         ┆ 60Z          ┆          ┆              ┆   ┆        ┆           ┆         ┆            │
│ 2       ┆ 2025-12-03T1 ┆ 7989747  ┆ set_slice_at ┆ … ┆ null   ┆ null      ┆ null    ┆ null       │
│         ┆ 2:19:12.2518 ┆          ┆ trs          ┆   ┆        ┆           

[{'version': 1,
  'ts_utc': '2025-12-03T12:19:12.251660Z',
  'mono_ns': 7807091,
  'op': 'set_slice_attrs',
  'slice_id': 'Healthy',
  'attrs': {'condition': 'Healthy'},
  'result': None},
 {'version': 2,
  'ts_utc': '2025-12-03T12:19:12.251856Z',
  'mono_ns': 7989747,
  'op': 'set_slice_attrs',
  'slice_id': 'Stressed',
  'attrs': {'condition': 'Stressed'},
  'result': None},
 {'version': 3,
  'ts_utc': '2025-12-03T12:19:12.251998Z',
  'mono_ns': 8130997,
  'op': 'set_slice_attrs',
  'slice_id': 'Disease',
  'attrs': {'condition': 'Disease'},
  'result': None}]

In [20]:
import sys
import pathlib

# add repo root to Python path
sys.path.append(str(pathlib.Path.cwd().parent))

In [21]:
csv1_path = 'csv1_edges.csv'
pl.DataFrame(
    {
        'source': ['A', 'A', 'B', 'C', 'D'],
        'target': ['B', 'C', 'C', 'D', 'A'],
        'weight': [1, 2, 3, 1, 5],
        'directed': [True, True, False, True, True],
        'slice': ['L1', 'L1', 'L1', 'L2', 'L2'],
    }
)  # .write_csv(csv1_path)

# pl.read_csv(csv1_path).head()

source,target,weight,directed,slice
str,str,i64,bool,str
"""A""","""B""",1,true,"""L1"""
"""A""","""C""",2,true,"""L1"""
"""B""","""C""",3,false,"""L1"""
"""C""","""D""",1,true,"""L2"""
"""D""","""A""",5,true,"""L2"""


In [ ]:
"""G = graph_csv.load_csv_to_graph(
    csv1_path,
    schema="auto",            # or 'edge_list'/'incidence'/'adjacency'/'hyperedge'/'lil'
    default_slice=None,       # fallback if no slice column is present
    default_directed=None,    # fallback if no directed column and cannot infer
    default_weight=1.0,
)

# Quick sanity: show first rows of an edges view (columns depend on your AnnNet implementation)
edges = G.views.edges(slice=None, include_directed=True, resolved_weight=True)
edges.head()
"""

In [ ]:
# Count entities and edges (global_entity_count and global_edge_count are methods)
num_entities = G.global_entity_count()  # vertices + edge-entities
num_edges = G.global_edge_count()  # binary + hyper

print('entities:', num_entities, 'edges:', num_edges)

# A light "degree" summary from edges_view for binary edges only (skip hyper)
df = G.views.edges(include_directed=True, resolved_weight=True)

# Try to find endpoint column names robustly
cols = {c.lower(): c for c in df.columns}
# Common possibilities:
src_col = next((cols[c] for c in ['source', 'src', 'u', 'from']), None)
dst_col = next((cols[c] for c in ['target', 'dst', 'v', 'to']), None)

if src_col and dst_col:
    # out-degree (directed) / degree (undirected)
    out_deg = df.group_by(src_col).len().rename({src_col: 'vertex', 'len': 'out_degree'})
    in_deg = df.group_by(dst_col).len().rename({dst_col: 'vertex', 'len': 'in_degree'})
    deg = out_deg.join(in_deg, on='vertex', how='full').fill_null(0)
    deg = deg.with_columns((pl.col('out_degree') + pl.col('in_degree')).alias('total_degree'))
    deg.sort('total_degree', descending=True).head(10)
else:
    print(
        'Skip degree summary: endpoint columns not found in edges_view output (likely hyperedge-only or different schema).'
    )

In [ ]:
# Add a new vertex and an edge on slice L3
G.add_vertices('E')
eid = G.add_edges('E', 'A', slice='L3', directed=True, weight=2.5)

# Per-slice weight override example:
G.attrs.set_edge_slice_attrs('L3', eid, weight=3.0)

# Inspect the updated edges
G.views.edges(include_directed=True, resolved_weight=True).tail()

In [ ]:
csv2_path = 'csv2_edges_view.csv'
G.views.edges(slice=None, include_directed=True, resolved_weight=True)
csv2_path

In [26]:
def export_edge_list_csv(G, path, slice=None):
    df = G.views.edges(slice=slice, include_directed=True, resolved_weight=True)
    cols = {c.lower(): c for c in df.columns}
    src_col = next((cols[c] for c in ['source', 'src', 'u', 'from']), None)
    dst_col = next((cols[c] for c in ['target', 'dst', 'v', 'to']), None)
    dir_col = next((cols[c] for c in ['directed']), None)
    w_eff = next((cols[c] for c in ['effective_weight', 'weight', 'w']), None)

    if not (src_col and dst_col):
        raise ValueError(
            'No binary endpoint columns found; the view may be hyperedge-only. Try the generic edges_view export.'
        )

    return pl.DataFrame(
        {
            'source': df[src_col],
            'target': df[dst_col],
            'weight': df[w_eff] if w_eff else pl.Series([1.0] * df.height),
            'directed': df[dir_col] if dir_col in df.columns else pl.Series([None] * df.height),
            'slice': pl.Series([slice] * df.height) if slice else pl.Series([None] * df.height),
        }
    )


# Usage:
csv2_edge_list_path = 'csv2_edge_list.csv'
export_edge_list_csv(G, csv2_edge_list_path, slice=None)
csv2_edge_list_path

'csv2_edge_list.csv'

In [27]:
# In-memory look at last few events
hist = G.history(as_df=True)  # DF [DataFrame]
hist.tail()

# Save to Parquet/CSV/JSON [JavaScript Object Notation]/NDJSON [Newline-Delimited JSON]
G.history.export('graph_history.parquet')

2162

In [ ]:
demo_path = 'ppi.annnet'
G.layers.set_aspects(['time', 'relation'])
G.layers.set_elementary_layers({'time': ['t1', 't2'], 'relation': ['F', 'A']})

In [29]:
G.write(demo_path, overwrite=True)  # lossless save
print('Wrote:', demo_path)

Wrote: ppi.annnet


In [30]:
import sys
import pathlib

sys.path.append(str(pathlib.Path.cwd().parent))

from annnet.core.graph import AnnNet

In [31]:
# G = graph_excel.load_excel_to_graph("graph_input.xlsx", schema="auto")

In [32]:
G.global_entity_count()

217

In [33]:
from annnet.adapters.networkx_adapter import to_nx

In [34]:
nxG, man = to_nx(G, directed=True, hyperedge_mode='skip')

In [35]:
from annnet.adapters.igraph_adapter import to_igraph

In [36]:
nxG, man = to_igraph(G, directed=True, hyperedge_mode='skip', public_only=False)

In [37]:
G.shape

(207, 1189)

In [ ]:
# Deterministic smoke tests for the lazy NX (NetworkX) proxy

# ---------- G1: PATH GRAPH (for weighted/unweighted shortest paths) ----------
def build_ig_path_graph() -> AnnNet:
    """Directed chain a→b→c→d→e→f with weights on each edge.
    - Weighted shortest path a→f = 1+2+3+1+4 = 11
    - Unweighted hops a→f = 5
    """
    G = AnnNet(directed=True)
    G.add_vertices('a', name='alpha')
    G.add_vertices('b', name='bravo')
    G.add_vertices('c', name='charlie')
    G.add_vertices('d', name='delta')
    G.add_vertices('e', name='echo')
    G.add_vertices('f', name='phi')

    G.add_edges('a', 'b', weight=1)
    G.add_edges('b', 'c', weight=2)
    G.add_edges('c', 'd', weight=3)
    G.add_edges('d', 'e', weight=1)
    G.add_edges('e', 'f', weight=4)

    return G


# ---------- G2: COMMUNITY GRAPH (two cliques + weak bridge) ----------
def build_ig_community_graph() -> AnnNet:
    G = AnnNet(directed=True)

    for v in ['a', 'b', 'c', 'd', 'e', 'f', 'w', 'x', 'y', 'z']:
        G.add_vertices(v)

    k6 = ['a', 'b', 'c', 'd', 'e', 'f']
    for i in range(len(k6)):
        for j in range(i + 1, len(k6)):
            G.add_edges(k6[i], k6[j], weight=1, directed=False)

    k4 = ['w', 'x', 'y', 'z']
    for i in range(len(k4)):
        for j in range(i + 1, len(k4)):
            G.add_edges(k4[i], k4[j], weight=1, directed=False)

    G.add_edges('e', 'x', weight=0.01, directed=False)

    return G


def run_tests():
    G1 = build_path_graph()

    dist_w = G1.nx.shortest_path_length(
        G1, source='alpha', target='phi', weight='weight', _nx_label_field='name'
    )
    print('[G1:dijkstra weighted] alpha->phi:', dist_w, '(expect 11.0)')

    dist_hops = G1.nx.shortest_path_length(G1, source='a', target='f', weight=None)
    print('[G1:unweighted hops] a->f:', dist_hops, '(expect 5)')

    G1.add_edges('a', 'f', weight=2)
    dist_new = G1.nx.shortest_path_length(G1, source='a', target='f', weight='weight')
    print('[G1:after mutation] a->f:', dist_new, '(expect 2.0)')

    G2 = build_community_graph()

    comms = G2.nx.louvain_communities(G2, _nx_directed=False, seed=42, weight='weight')
    sizes = sorted(len(c) for c in comms)
    print('[G2:louvain] sizes:', sizes, '(expect [4, 6])')

    bc = G2.nx.betweenness_centrality(G2, _nx_directed=False, normalized=True)
    top_bc = max(bc, key=bc.get)
    print('[G2:betweenness] top:', top_bc, "(expect 'e' or 'x')")

    pr = G2.nx.pagerank(G2, _nx_directed=False)
    top_pr = max(pr, key=pr.get)
    print('[G2:pagerank] top:', top_pr, "(expect 'e')")

    comps = list(G2.nx.connected_components(G2, _nx_directed=False))
    print(
        '[G2:connected components]:',
        [sorted(c) for c in comps],
        '(expect one component of size 10)',
    )


if __name__ == '__main__':
    run_tests()

In [ ]:
# Deterministic smoke tests for the lazy ig (igraph) proxy

from annnet.core.graph import AnnNet


def build_path_graph() -> AnnNet:
    """Directed chain a→b→c→d→e→f with weights.
    Weighted a→f = 11, unweighted hops = 5.
    """
    G = AnnNet(directed=True)
    G.add_vertices('a', name='alpha')
    G.add_vertices('b', name='bravo')
    G.add_vertices('c', name='charlie')
    G.add_vertices('d', name='delta')
    G.add_vertices('e', name='echo')
    G.add_vertices('f', name='phi')

    G.add_edges('a', 'b', weight=1)
    G.add_edges('b', 'c', weight=2)
    G.add_edges('c', 'd', weight=3)
    G.add_edges('d', 'e', weight=1)
    G.add_edges('e', 'f', weight=4)
    return G


def build_community_graph() -> AnnNet:
    """Two undirected cliques K6+K4 joined by a weak bridge e--x."""
    G = AnnNet(directed=True)

    for v in ['a', 'b', 'c', 'd', 'e', 'f', 'w', 'x', 'y', 'z']:
        G.add_vertices(v)

    k6 = ['a', 'b', 'c', 'd', 'e', 'f']
    for i in range(len(k6)):
        for j in range(i + 1, len(k6)):
            G.add_edges(k6[i], k6[j], weight=1, directed=False)

    k4 = ['w', 'x', 'y', 'z']
    for i in range(len(k4)):
        for j in range(i + 1, len(k4)):
            G.add_edges(k4[i], k4[j], weight=1, directed=False)

    G.add_edges('e', 'x', weight=0.01, directed=False)
    return G


def _unwrap_ig_distance(obj):
    if isinstance(obj, (list, tuple)) and len(obj) == 1:
        inner = obj[0]
        if isinstance(inner, (list, tuple)) and len(inner) == 1:
            return inner[0]
        return inner
    return obj


def run_tests():
    G1 = build_path_graph()

    dist_w = G1.ig.shortest_paths_dijkstra(
        source='alpha', target='phi', weights='weight', _ig_guess_labels=False
    )
    dist_w = _unwrap_ig_distance(dist_w)
    print('[IG:G1 dijkstra weighted] alpha->phi:', dist_w, '(expect 11.0)')

    dist_hops = G1.ig.distances(source='alpha', target='phi', weights=None, _ig_guess_labels=False)
    dist_hops = _unwrap_ig_distance(dist_hops)
    print('[IG:G1 unweighted hops] alpha->phi:', dist_hops, '(expect 5)')

    G1.add_edges('a', 'f', weight=2)
    dist_new = G1.ig.shortest_paths_dijkstra(
        source='alpha', target='phi', weights='weight', _ig_guess_labels=False
    )
    dist_new = _unwrap_ig_distance(dist_new)
    print('[IG:G1 after mutation] alpha->phi:', dist_new, '(expect 2.0)')

    G2 = build_community_graph()

    vc = G2.ig.community_multilevel(weights='weight', _ig_directed=False)
    sizes = sorted(vc.sizes())
    print('[IG:G2 multilevel] sizes:', sizes, '(expect [4, 6])')

    igG_und = G2.ig.backend(directed=False)
    names = (
        igG_und.vs['name'] if 'name' in igG_und.vs.attributes() else list(range(igG_und.vcount()))
    )
    bc_vals = G2.ig.betweenness(directed=False, weights=None)
    top_bc = max(
        dict(zip(names, bc_vals, strict=False)),
        key=lambda k: dict(zip(names, bc_vals, strict=False))[k],
    )
    print('[IG:G2 betweenness] top:', top_bc, "(expect 'e' or 'x')")

    pr_vals = G2.ig.pagerank(directed=False)
    top_pr = max(
        dict(zip(names, pr_vals, strict=False)),
        key=lambda k: dict(zip(names, pr_vals, strict=False))[k],
    )
    print('[IG:G2 pagerank] top:', top_pr, "(expect 'e')")

    comps = G2.ig.components(_ig_directed=False)
    comp_sizes = sorted(comps.sizes())
    print('[IG:G2 connected components]:', comp_sizes, '(expect [10])')

    G3 = AnnNet(directed=True)
    G3.add_vertices('u')
    G3.add_vertices('v')
    G3.add_edges('u', 'v', weight=5, capacity=3, directed=False)
    G3.add_edges('u', 'v', weight=2, capacity=7, directed=False)

    try:
        igG_simple = G3.ig.backend(
            directed=False,
            simple=True,
            needed_attrs={'weight', 'capacity'},
            edge_aggs={'weight': 'min', 'capacity': 'sum'},
        )
        e = igG_simple.es[0]
        print(
            '[IG:G3 collapse agg via proxy] weight:',
            e['weight'],
            'capacity:',
            e['capacity'],
            '(expect 2, 10)',
        )
    except Exception:
        igG_raw = G3.ig.backend(directed=False, needed_attrs={'weight', 'capacity'})
        igG_raw.simplify(
            multiple=True, loops=True, combine_edges={'weight': 'min', 'capacity': 'sum'}
        )
        e = igG_raw.es[0]
        w = e['weight'] if 'weight' in igG_raw.es.attributes() else None
        c = e['capacity'] if 'capacity' in igG_raw.es.attributes() else None
        print('[IG:G3 collapse agg via simplify] weight:', w, 'capacity:', c, '(expect 2, 10)')


if __name__ == '__main__':
    run_tests()